# 6.1: Building a RAG-based Q&A System using Hugging Face and the SQuAD v2 Dataset
## Marco Antonio Gonzalez

In [1]:
!pip install datasets sentence-transformers faiss-cpu openai -q


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: pip install --upgrade pip


## 1. Load and Explore SQuAD v2 Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("squad_v2")
print(dataset["train"][0])

{'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'question': 'When did Beyonce start becoming popular?', 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}


In [3]:
print("Dataset structure:")
print(dataset)
print(f"\nTotal training examples: {len(dataset['train'])}")
print(f"Total validation examples: {len(dataset['validation'])}")

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

Total training examples: 130319
Total validation examples: 11873


In [4]:
train_data = dataset["train"]
sample = train_data[0]

print(f"ID: {sample['id']}")
print(f"\nTitle: {sample['title']}")
print(f"\nContext: {sample['context'][:300]}...")
print(f"\nQuestion: {sample['question']}")
print(f"\nAnswers: {sample['answers']}")

ID: 56be85543aeaaa14008c9063

Title: Beyoncé

Context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singe...

Question: When did Beyonce start becoming popular?

Answers: {'text': ['in the late 1990s'], 'answer_start': [269]}


## 2. Implement Semantic Retriever

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

2025-10-15 09:06:20.078295: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-15 09:06:20.114029: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-15 09:06:20.899115: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Embedding dimension: 384


In [6]:
subset_size = 5000
contexts = [train_data[i]['context'] for i in range(subset_size)]
metadata = [{
    'question': train_data[i]['question'],
    'answers': train_data[i]['answers'],
    'title': train_data[i]['title']
} for i in range(subset_size)]

print(f"Processing {len(contexts)} contexts...")

Processing 5000 contexts...


In [7]:
context_embeddings = model.encode(contexts, show_progress_bar=True, batch_size=32)
print(f"Embeddings shape: {context_embeddings.shape}")

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)


In [8]:
dimension = context_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(context_embeddings.astype('float32'))
print(f"FAISS index created with {index.ntotal} vectors")

FAISS index created with 5000 vectors


In [9]:
def retrieve_context(query, top_k=3):
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    
    retrieved = []
    for idx, dist in zip(indices[0], distances[0]):
        retrieved.append({
            'context': contexts[idx],
            'distance': float(dist),
            'index': int(idx),
            'metadata': metadata[idx]
        })
    return retrieved

In [10]:
test_query = "What is the capital of France?"
results = retrieve_context(test_query, top_k=2)

print(f"Query: {test_query}\n")
for i, result in enumerate(results):
    print(f"Result {i+1} (distance: {result['distance']:.2f}):")
    print(f"Title: {result['metadata']['title']}")
    print(f"Context: {result['context'][:200]}...\n")

Query: What is the capital of France?

Result 1 (distance: 1.41):
Title: New_York_City
Context: The first documented visit by a European was in 1524 by Giovanni da Verrazzano, a Florentine explorer in the service of the French crown, who sailed his ship La Dauphine into New York Harbor. He claim...

Result 2 (distance: 1.41):
Title: New_York_City
Context: The first documented visit by a European was in 1524 by Giovanni da Verrazzano, a Florentine explorer in the service of the French crown, who sailed his ship La Dauphine into New York Harbor. He claim...



## 3. Integrate LM Studio LLM for Answer Generation

In [11]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

In [12]:
def generate_answer(query, contexts):
    context_text = "\n\n".join([f"Context {i+1}: {ctx['context']}" for i, ctx in enumerate(contexts)])
    
    prompt = f"""Based on the following contexts, answer the question. If the answer is not in the contexts, say "I cannot answer based on the given context."

{context_text}

Question: {query}
Answer:"""
    
    response = client.chat.completions.create(
        model="gemma-3-27b-it-qat",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions based on provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=200
    )
    
    return response.choices[0].message.content

## 4. Complete RAG Pipeline

In [13]:
def rag_pipeline(query, top_k=3):
    print(f"Query: {query}\n")
    print("=" * 80)
    
    retrieved_contexts = retrieve_context(query, top_k=top_k)
    print(f"\nRetrieved {len(retrieved_contexts)} contexts\n")
    
    for i, ctx in enumerate(retrieved_contexts):
        print(f"Context {i+1} (distance: {ctx['distance']:.2f}):")
        print(f"Title: {ctx['metadata']['title']}")
        print(f"Text: {ctx['context'][:300]}...\n")
    
    print("=" * 80)
    answer = generate_answer(query, retrieved_contexts)
    print(f"\nGenerated Answer:\n{answer}")
    print("=" * 80)
    
    return {
        'query': query,
        'retrieved_contexts': retrieved_contexts,
        'answer': answer
    }

## 5. Qualitative Evaluation

In [23]:
test_questions = [
    "Who is Beyonce",
    "Who invented the telephone?",
    "What produces a rainbow?",
    "Give me a plot of To Kill a Mockingbird?",
    "What is photosynthesis?"
]

results = []
for question in test_questions:
    print("\n" + "#" * 80)
    result = rag_pipeline(question, top_k=3)
    results.append(result)
    print()


################################################################################
Query: Who is Beyonce


Retrieved 3 contexts

Context 1 (distance: 0.77):
Title: Beyoncé
Text: Beyoncé Giselle Knowles was born in Houston, Texas, to Celestine Ann "Tina" Knowles (née Beyincé), a hairdresser and salon owner, and Mathew Knowles, a Xerox sales manager. Beyoncé's name is a tribute to her mother's maiden name. Beyoncé's younger sister Solange is also a singer and a former member ...

Context 2 (distance: 0.77):
Title: Beyoncé
Text: Beyoncé Giselle Knowles was born in Houston, Texas, to Celestine Ann "Tina" Knowles (née Beyincé), a hairdresser and salon owner, and Mathew Knowles, a Xerox sales manager. Beyoncé's name is a tribute to her mother's maiden name. Beyoncé's younger sister Solange is also a singer and a former member ...

Context 3 (distance: 0.77):
Title: Beyoncé
Text: Beyoncé Giselle Knowles was born in Houston, Texas, to Celestine Ann "Tina" Knowles (née Beyincé), a hairdresser an

In [26]:
print("QUALITATIVE EVALUATION ANALYSIS")
print("=" * 80)
print("\nStrengths:")
print("- Semantic retrieval effectively finds relevant contexts using sentence embeddings")
print("- FAISS provides fast similarity search even with large datasets")
print("- LM Studio integration allows local LLM inference without API costs")
print("- Pipeline successfully combines retrieval and generation")

print("\nLimitations:")
print("- Answer quality depends on retrieval accuracy")
print("- Limited to contexts present in SQuAD v2 dataset")
print("- Top-k parameter requires tuning for optimal results")
print("- No re-ranking mechanism for retrieved contexts")
print("- Simple L2 distance may not capture semantic similarity optimally")



QUALITATIVE EVALUATION ANALYSIS

Strengths:
- Semantic retrieval effectively finds relevant contexts using sentence embeddings
- FAISS provides fast similarity search even with large datasets
- LM Studio integration allows local LLM inference without API costs
- Pipeline successfully combines retrieval and generation

Limitations:
- Answer quality depends on retrieval accuracy
- Limited to contexts present in SQuAD v2 dataset
- Top-k parameter requires tuning for optimal results
- No re-ranking mechanism for retrieved contexts
- Simple L2 distance may not capture semantic similarity optimally


In [25]:
print("\nEXAMPLE ANALYSIS:")
print("=" * 80)
for i, result in enumerate(results[:3]):
    print(f"\nExample {i+1}:")
    print(f"Question: {result['query']}")
    print(f"\nRetrieved Context Quality:")
    print(f"- Number of contexts: {len(result['retrieved_contexts'])}")
    print(f"- Closest match distance: {result['retrieved_contexts'][0]['distance']:.2f}")
    print(f"\nAnswer: {result['answer']}")
    print(f"\nRelevance Assessment:")
    if result['retrieved_contexts'][0]['distance'] < 50:
        print("- High relevance: Retrieved context closely matches query")
    elif result['retrieved_contexts'][0]['distance'] < 100:
        print("- Medium relevance: Retrieved context partially matches query")
    else:
        print("- Low relevance: Retrieved context may not be ideal")
    print("-" * 80)


EXAMPLE ANALYSIS:

Example 1:
Question: Who is Beyonce

Retrieved Context Quality:
- Number of contexts: 3
- Closest match distance: 0.77

Answer: Beyoncé Giselle Knowles was born in Houston, Texas, to Celestine Ann "Tina" Knowles (née Beyincé), a hairdresser and salon owner, and Mathew Knowles, a Xerox sales manager. Her name is a tribute to her mother's maiden name. She has a younger sister named Solange who is also a singer and a former member of Destiny's Child.

Relevance Assessment:
- High relevance: Retrieved context closely matches query
--------------------------------------------------------------------------------

Example 2:
Question: Who invented the telephone?

Retrieved Context Quality:
- Number of contexts: 3
- Closest match distance: 1.40

Answer: I cannot answer based on the given context. The contexts discuss Kane Kramer's invention of a "plastic music box" called the IXI, not the telephone.

Relevance Assessment:
- High relevance: Retrieved context closely matches 